# Testing the worm

In this notebook, we implement several interactive tests that aim at checking that the worm algorithm we implemented samples correctly from the conditional distribution

In [ ]:
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from jax.typing import ArrayLike
import jax.numpy as jnp
from functools import partial
import numpy as np
from typing import Dict
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_coherent_information,
    run_worm
)
import time
import json
import os
import jax
N_CPUS = os.cpu_count()
N_USED_CPUS = N_CPUS
print("Number of CPUs available: {}".format(N_CPUS))
print("Number of used CPUs: {}".format(N_USED_CPUS))
jax.config.update('jax_num_cpu_devices', N_USED_CPUS)
# jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


# Test independency of worm error start

The first test consists in testing that the conditional probabilities do not depend on the worm start, if we take the burn in steps high enough. Note that this also means that you could start the worm from the candidate error, which is in my opinion an option to turn the algorithm into a decoder. 

In [ ]:
length = 5
width = length
p = 3
num_samples = 1
num_worms = 400 * N_USED_CPUS  # 5000
burn_in_steps = length * 15000  # 50000
max_worm_steps = burn_in_steps + length * 10000
gamma_t = 0.3

In [16]:
moebius_code = MoebiusCodeTwoOddPrime(
    length=length, width=width, d=2 * p)
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * p, gamma_t=gamma_t
)
print(f"Single error probabilities: {em_lindblad.get_probabilities()}")

Single error probabilities: [0.59932834 0.1721758  0.02563445 0.00505141 0.02563424 0.1721757 ]


In [17]:
master_error_key = jax.random.PRNGKey(144)

initial_error_mod_d = em_lindblad.generate_random_error(master_error_key)
initial_error_mod_2 = jnp.mod(initial_error_mod_d, 2)
initial_error_mod_p = jnp.mod(initial_error_mod_d, p)
initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))

In [18]:
h_mod_2 = moebius_code.h_x_mod_2
h_mod_p = moebius_code.h_x_mod_p
h_error_mod_p = moebius_code.h_z_mod_p

syndrome_mod_2 = jnp.mod(h_mod_2 @ initial_error_mod_2, 2)
syndrome_mod_p = jnp.mod(h_mod_p @ initial_error_mod_p, p)
syndrome = jnp.vstack([syndrome_mod_2, syndrome_mod_p])
syndrome_mod_d = jnp.mod(moebius_code.h_x @ initial_error_mod_d, 2 * p)

compute_chi = moebius_code.compute_plaquette_syndrome_chi_x
initial_chi = jnp.mod(compute_chi(initial_error_mod_d)[-1], p)
num_stabs = h_mod_p.shape[0]

print(f"Initial chi: {initial_chi}")

Initial chi: 0


In [19]:
run_worm_partial = partial(
    run_worm,
    h_error_mod_p=h_error_mod_p,
    h_mod_p=h_mod_p,
    error_model=em_lindblad,
    compute_full_chi=compute_chi,
    num_stabs=num_stabs,
    burn_in_steps=burn_in_steps,
    max_worm_steps=max_worm_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
# run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, 0))
run_worm_jit = jax.jit(run_worm_vmap)

In [20]:
def get_chi_probs_and_binary_entropy(chi_vec, success_vec):
    # Number of successful worms
    num_success = jnp.sum(success_vec)
    # Sets simply to zero failed attempts so that they are not counted
    chi_vec_marked = jnp.where(success_vec, chi_vec, 0)
    p1 = jnp.sum(chi_vec_marked) / num_success
    p0 = 1 - p1
    binary_entropy = -jax.scipy.special.xlogy(p0, p0) / jnp.log(2)
    binary_entropy += -jax.scipy.special.xlogy(p1, p1) / jnp.log(2)
    return p0, p1, binary_entropy

In [ ]:
master_worm_key = jax.random.PRNGKey(1334)
worm_keys = jax.random.split(master_worm_key, num_worms)

devices = jax.devices()  # Assuming this returns 16 devices
mesh = Mesh(devices, ('batch',))

# sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
# worm_keys_sharded = jax.device_put(worm_keys, sharding_for_keys)

# 2. Define sharding: Split the 0th axis across 'batch', leave others whole
sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
worm_keys_sharded = jax.device_put(
    worm_keys, sharding_for_keys)

new_worm_state = run_worm_jit(initial_worm_error, worm_keys_sharded)

p0, p1, entropy = get_chi_probs_and_binary_entropy(
    new_worm_state["chi"], new_worm_state["worm_success"])

In [22]:
# Now we try to do the same but using the candidate error

candidate_error_mod_d = moebius_code.get_plaquette_candidate_error(
    syndrome_mod_d)
candidate_error_mod_2 = jnp.mod(candidate_error_mod_d, 2)
candidate_error_mod_p = jnp.mod(candidate_error_mod_d, 2 * p)
candidate_error = jnp.vstack([candidate_error_mod_2, candidate_error_mod_p])

In [ ]:
master_worm_key_candidate = jax.random.PRNGKey(14534)
worm_keys_candidate = jax.random.split(master_worm_key_candidate, num_worms)

worm_keys_candidate_sharded = jax.device_put(
    worm_keys_candidate, sharding_for_keys)

# Now we try to do the same but using the candidate error

candidate_error_mod_d = moebius_code.get_plaquette_candidate_error(
    syndrome_mod_d)
candidate_error_mod_2 = jnp.mod(candidate_error_mod_d, 2)
candidate_error_mod_p = jnp.mod(candidate_error_mod_d, 2 * p)
candidate_error = jnp.vstack([candidate_error_mod_2, candidate_error_mod_p])

new_worm_state_candidate = run_worm_jit(
    candidate_error, worm_keys_candidate_sharded)

p0_candidate, p1_candidate, entropy_candidate = get_chi_probs_and_binary_entropy(
    new_worm_state_candidate["chi"],
    new_worm_state_candidate["worm_success"]
)

In [24]:
# We also compute the probability starting from an error that gives a different
# logical bit chi compared to the initial one

master_worm_key_test = jax.random.PRNGKey(65)
worm_keys_test = jax.random.split(master_worm_key_test, num_worms)

worm_keys_test_sharded = jax.device_put(
    worm_keys_test, sharding_for_keys)

different_chi_locations = jnp.where(new_worm_state["chi"] != initial_chi)

test_error = new_worm_state["worm_error"][different_chi_locations[0][0], :]

new_worm_state_test = run_worm_jit(test_error, worm_keys_test_sharded)

p0_test, p1_test, entropy_test = get_chi_probs_and_binary_entropy(
    new_worm_state_test["chi"],
    new_worm_state_test["worm_success"]
)

In [25]:
num_success = jnp.sum(new_worm_state["worm_success"])
num_success_candidate = jnp.sum(new_worm_state_candidate["worm_success"])
num_success_test = jnp.sum(new_worm_state_test["worm_success"])

print(f"Probability chi 0: {p0}")
print(f"Probability chi 0 candidate: {p0_candidate}")
print(f"Probability chi 0 test: {p0_test}")
print("----------------------------")
print(f"Entropy: {entropy}")
print(f"Entropy candidate: {entropy_candidate}")
print(f"Entropy test: {entropy_test}")
print("----------------------------")
print(f"Number of successes: {num_success}")
print(f"Number of successes candidate: {num_success_candidate}")
print(f"Number of successes test: {num_success_test}")

Probability chi 0: 0.9478124976158142
Probability chi 0 candidate: 0.9498437643051147
Probability chi 0 test: 0.9471792578697205
----------------------------
Entropy: 0.29561761021614075
Entropy candidate: 0.28706032037734985
Entropy test: 0.29826056957244873
----------------------------
Number of successes: 6400
Number of successes candidate: 6400
Number of successes test: 6399


In [26]:
-p0_test * np.log2(p0_test) - (1 - p0_test) * np.log2(1 - p0_test)

Array(0.2982605, dtype=float32)